# global-wheat-detection → 350×350 确定性切块（写入 `EWS/data/pretrain/npy`）

- **数据**：`EWS/data/global-wheat-detection/train/` 与 `test/` 下 `*.jpg`（均为 **1024×1024** RGB）。
- **策略**：与 sugar-beets 相同思路——**无随机起点**；最少 **350×350** patch **无重叠** 铺满全图：
  - \(n_y=n_x=\lceil 1024/350\rceil=3\)，每图 **9** 块。
  - 画布 **1050×1050**（仅 **下、右** 各补 **26** 像素零）。
- **输出**：每个 patch 一个 `.npy`，**直接**写入已存在的 **`EWS/data/pretrain/npy/`**（与 sugar-beets 共用；**不会 `mkdir`**，目录须事先建好）。
- **命名**：`globalwheat_{train|test}_{原jpg主名}__r{行}_c{列}.npy`，避免与其它数据集 stem 冲突。
- **数组**：`float32`，`(350, 350, 3)`，\([0,1]\)。
- **进度**：`tqdm` 总进度条；**每 100 张图**一行 `tqdm.write`。需 `pip install tqdm`（若未安装）。

In [4]:
from pathlib import Path

import numpy as np
from PIL import Image

PATCH = 350
H0, W0 = 1024, 1024
N_ROWS = (H0 + PATCH - 1) // PATCH
N_COLS = (W0 + PATCH - 1) // PATCH
CANVAS_H = N_ROWS * PATCH
CANVAS_W = N_COLS * PATCH
PAD_BOTTOM = CANVAS_H - H0
PAD_RIGHT = CANVAS_W - W0

print(
    f"网格: {N_ROWS}×{N_COLS} = {N_ROWS * N_COLS} patches / 图; "
    f"画布 {CANVAS_H}×{CANVAS_W}; pad下={PAD_BOTTOM}, 右={PAD_RIGHT}"
)

网格: 3×3 = 9 patches / 图; 画布 1050×1050; pad下=26, 右=26


In [3]:
def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        gw = p / "EWS" / "data" / "global-wheat-detection"
        if (gw / "train").is_dir() and (gw / "test").is_dir():
            return p
    raise FileNotFoundError(
        "未找到 global-wheat-detection。请在仓库根 AR-SSL4M 下启动内核，或把 cwd 切到该仓库。"
    )


REPO = find_repo(Path.cwd())
GW_ROOT = REPO / "EWS" / "data" / "global-wheat-detection"
TRAIN_DIR = GW_ROOT / "train"
TEST_DIR = GW_ROOT / "test"
OUT_DIR = REPO / "EWS" / "data" / "pretrain" / "npy"
if not OUT_DIR.is_dir():
    raise FileNotFoundError(
        f"输出目录必须已存在（不自动创建），请将 .npy 放在: {OUT_DIR}"
    )

print("REPO:", REPO)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)
print("OUT_DIR:", OUT_DIR)

REPO: D:\Cursor\AR-SSL4M
TRAIN_DIR: D:\Cursor\AR-SSL4M\EWS\data\global-wheat-detection\train
TEST_DIR: D:\Cursor\AR-SSL4M\EWS\data\global-wheat-detection\test
OUT_DIR: D:\Cursor\AR-SSL4M\EWS\data\pretrain\npy


In [5]:
def image_to_padded_canvas(arr_hwc: np.ndarray) -> np.ndarray:
    if arr_hwc.shape[0] != H0 or arr_hwc.shape[1] != W0:
        raise ValueError(f"期望 H,W = {H0},{W0}, 得到 {arr_hwc.shape[:2]}")
    return np.pad(
        arr_hwc,
        ((0, PAD_BOTTOM), (0, PAD_RIGHT), (0, 0)),
        mode="constant",
        constant_values=0,
    )


def save_patches_for_image(img_path: Path, split: str) -> list[Path]:
    """split: 'train' | 'test'，写入文件名前缀。"""
    stem = img_path.stem
    prefix = f"globalwheat_{split}_{stem}"
    im = Image.open(img_path).convert("RGB")
    arr = np.asarray(im, dtype=np.uint8)
    canvas = image_to_padded_canvas(arr)
    saved = []
    for r in range(N_ROWS):
        for c in range(N_COLS):
            y0, y1 = r * PATCH, (r + 1) * PATCH
            x0, x1 = c * PATCH, (c + 1) * PATCH
            patch = canvas[y0:y1, x0:x1, :].astype(np.float32) / 255.0
            out_path = OUT_DIR / f"{prefix}__r{r}_c{c}.npy"
            np.save(out_path, patch, allow_pickle=False)
            saved.append(out_path)
    return saved

In [6]:
from tqdm.auto import tqdm

train_paths = sorted(TRAIN_DIR.glob("*.jpg"))
test_paths = sorted(TEST_DIR.glob("*.jpg"))
tasks = [(p, "train") for p in train_paths] + [(p, "test") for p in test_paths]

print(
    f"train {len(train_paths)} 张, test {len(test_paths)} 张, "
    f"合计 {len(tasks)} 张；每张 {N_ROWS * N_COLS} 个 npy"
)

all_saved = []
for i, (p, split) in enumerate(tqdm(tasks, desc="切分 global-wheat", unit="图")):
    paths = save_patches_for_image(p, split)
    all_saved.extend(paths)
    if (i + 1) % 100 == 0:
        tqdm.write(
            f"已完成 {i + 1} 张图（{split} / {p.name}），累计 {len(all_saved)} 个 npy"
        )

print(f"全部完成，共写入 {len(all_saved)} 个 .npy（均在 {OUT_DIR} 下）")

c:\Users\hrole\.conda\envs\dataset-tools\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train 3422 张, test 10 张, 合计 3432 张；每张 9 个 npy


切分 global-wheat:   3%|▎         | 104/3432 [00:02<01:15, 43.95图/s]  

已完成 100 张图（train / 07dc460bc.jpg），累计 900 个 npy


切分 global-wheat:   6%|▌         | 204/3432 [00:04<01:14, 43.30图/s]   

...

已完成 3300 张图（train / f6d969b95.jpg），累计 29700 个 npy


切分 global-wheat:  99%|█████████▉| 3404/3432 [03:22<00:01, 17.37图/s]   

已完成 3400 张图（train / fe5c215e2.jpg），累计 30600 个 npy


切分 global-wheat: 100%|██████████| 3432/3432 [03:24<00:00, 16.82图/s]

全部完成，共写入 30888 个 .npy（均在 D:\Cursor\AR-SSL4M\EWS\data\pretrain\npy 下）
